# 0. Complete SafeDrive Colab driver

This is the single canonical notebook. The final question is whether geometry-competent SAC can adapt to simulator-controlled procedural traffic, reduce collisions, and retain traffic-free generalization. The learned policy controls one ego vehicle only (`num_agents=1`, `is_multi_agent=false`). Run exactly two seed-0 pilots and one selected seed-1 confirmation; do not add seeds or variants automatically.

## 1. Constants and paths

In [ ]:
import json
import os
from pathlib import Path

REPO_URL = "https://github.com/djdhillxn/safedrive.git"
REPO_DIR = Path("/content/safedrive")
DRIVE_PROJECT = Path("/content/drive/MyDrive/SafeDrive")
RUNS_DIR = REPO_DIR / "runs"
CPU_COUNT = os.cpu_count() or 2
N_ENVS = min(4, max(1, CPU_COUNT - 1))
VEC_ENV = "subproc" if N_ENVS > 1 else "dummy"
print({"CPU_COUNT": CPU_COUNT, "N_ENVS": N_ENVS, "VEC_ENV": VEC_ENV})

## 2. Runtime, CPU, RAM, CUDA, and L4 inspection

In [ ]:
import platform
import psutil
import torch
print(platform.platform())
print(f"CPU: {CPU_COUNT}; RAM: {psutil.virtual_memory().total / 2**30:.1f} GiB")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || true

## 3. Google Drive mount

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
DRIVE_PROJECT.mkdir(parents=True, exist_ok=True)

## 4. Clone or fast-forward pull repository

In [ ]:
if not (REPO_DIR / ".git").exists():
    !git clone --depth 1 {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} fetch origin main
    !git -C {REPO_DIR} merge --ff-only origin/main
%cd /content/safedrive

## 5. Install repository and verify package versions

In [ ]:
%pip uninstall -y gym
%pip install --disable-pip-version-check -e .
import gymnasium, metadrive, rich, stable_baselines3, tqdm
print("MetaDrive", getattr(metadrive, "__version__", "pinned source"))
print("SB3", stable_baselines3.__version__, "Gymnasium", gymnasium.__version__)
print("tqdm/rich available", tqdm.__version__, rich.__version__ if hasattr(rich, "__version__") else "yes")

## 6. Restore artifacts from Drive

In [ ]:
!python -m scripts.sync_drive_runs --drive-project "{DRIVE_PROJECT}" --local-runs "{RUNS_DIR}" --include-training-artifacts

def optional_latest(name):
    pointer = RUNS_DIR / f"latest_{name}.txt"
    if not pointer.exists():
        return None
    value = Path(pointer.read_text().strip())
    return value if value.is_absolute() else REPO_DIR / value

def run_status(run_dir):
    if run_dir is None:
        return "missing"
    state = run_dir / "curriculum_state.json"
    metadata = run_dir / "run_metadata.json"
    path = state if state.exists() else metadata
    return json.loads(path.read_text()).get("status", "unknown")

def training_action(pointer_name):
    run_dir = optional_latest(pointer_name)
    status = run_status(run_dir)
    if status == "complete":
        return "reuse", run_dir
    if status == "paused":
        return "resume", run_dir
    if status == "failed_gate":
        return "terminal", run_dir
    if status == "missing":
        return "start", None
    raise RuntimeError(f"Refusing ambiguous run status {status!r}: {run_dir}")

## 7. Compile, test, MetaDrive smoke test, and chase-camera smoke test

In [ ]:
!python -m compileall saferl_drive scripts tests
!pytest -q
!python -m scripts.evaluate_baseline --config configs/sac_traffic_curriculum.yaml --policy expert --split validation --episodes 1 --densities 0.05 --prefix traffic_smoke --no-progress validation.num_scenarios=1
!python -m scripts.record_video --config configs/sac_traffic_curriculum.yaml --policy expert --view chase --density 0.05 --seed 50000 --steps 5 --output /tmp/safedrive_chase_smoke.mp4
assert Path("/tmp/safedrive_chase_smoke.mp4").exists()
print("Compile, tests, traffic smoke, progress dependencies, and frame-producing chase smoke passed.")

## 8. Summarize existing Phase-1 and Phase-2 results without retraining

In [ ]:
import pandas as pd
for path in [RUNS_DIR / "phase1_comparison.csv", RUNS_DIR / "phase2_comparison.csv", RUNS_DIR / "phase2_light_traffic_comparison.csv"]:
    if path.exists():
        display(pd.read_csv(path))
print("Historical Phase 1/2 runs are evidence and are never retrained by this notebook.")

## 9. Experiment 0: IDM and ExpertPolicy traffic solvability

In [ ]:
!python -m scripts.evaluate_baseline --config configs/sac_traffic_curriculum.yaml --policy idm --split validation --episodes 50 --densities 0.05 0.10 --prefix traffic_solvability_idm validation.num_scenarios=50
!python -m scripts.evaluate_baseline --config configs/sac_traffic_curriculum.yaml --policy expert --split validation --episodes 50 --densities 0.05 0.10 --prefix traffic_solvability_expert validation.num_scenarios=50
!python -m scripts.record_video --config configs/sac_traffic_curriculum.yaml --policy expert --view chase --density 0.05 --seed 50000 --steps 250 --output runs/videos/expert_solubility_chase_d005_seed50000.mp4
for name in ["traffic_extension_idm", "traffic_extension_expert"]:
    baseline_run = optional_latest(name)
    !python -m scripts.sync_drive_runs --drive-project "{DRIVE_PROJECT}" --to-drive --run-dir "{baseline_run}"

## 10. Experiment 1: frozen pre-adaptation evaluation matrix

In [ ]:
for seed in [0, 1]:
    source_run = optional_latest(f"sac_phase2_curriculum_seed{seed}")
    if source_run is None:
        raise FileNotFoundError(f"Restore the completed curriculum seed-{seed} run from Drive.")
    !python -m scripts.evaluate --run-dir "{source_run}" --model best --split test --episodes 100 --densities 0.0 0.05 0.10 --prefix traffic_before
    !python -m scripts.sync_drive_runs --drive-project "{DRIVE_PROJECT}" --to-drive --run-dir "{source_run}"
SOURCE_SEED0_RUN = optional_latest("sac_phase2_curriculum_seed0")
!python -m scripts.evaluate --run-dir "{SOURCE_SEED0_RUN}" --model best --split validation --episodes 50 --densities 0.0 0.05 0.10 --prefix traffic_source_validation validation.num_scenarios=50
!python -m scripts.sync_drive_runs --drive-project "{DRIVE_PROJECT}" --to-drive --run-dir "{SOURCE_SEED0_RUN}"

## 11. Experiment 2: seed-0 SAC-Traffic pilot

In [ ]:
action, REFERENCE_RUN = training_action("sac_traffic_reference_seed0")
if action == "start":
    !python -m scripts.train_curriculum --config configs/sac_traffic_curriculum.yaml --variant reference --seed 0 --n-envs {N_ENVS} --vec-env {VEC_ENV} --progress
elif action == "resume":
    !python -m scripts.train_curriculum --config configs/sac_traffic_curriculum.yaml --variant reference --seed 0 --n-envs {N_ENVS} --vec-env {VEC_ENV} --progress --run-dir "{REFERENCE_RUN}"
else:
    print(f"Reference pilot action: {action}; run: {REFERENCE_RUN}")
REFERENCE_RUN = optional_latest("sac_traffic_reference_seed0")
!python -m scripts.sync_drive_runs --drive-project "{DRIVE_PROJECT}" --to-drive --run-dir "{REFERENCE_RUN}"

## 12. Experiment 3: seed-0 SAC-Traffic-Safe pilot

In [ ]:
action, SAFETY_RUN = training_action("sac_traffic_safety_seed0")
if action == "start":
    !python -m scripts.train_curriculum --config configs/sac_traffic_curriculum.yaml --variant safety --seed 0 --n-envs {N_ENVS} --vec-env {VEC_ENV} --progress
elif action == "resume":
    !python -m scripts.train_curriculum --config configs/sac_traffic_curriculum.yaml --variant safety --seed 0 --n-envs {N_ENVS} --vec-env {VEC_ENV} --progress --run-dir "{SAFETY_RUN}"
else:
    print(f"Safety pilot action: {action}; run: {SAFETY_RUN}")
SAFETY_RUN = optional_latest("sac_traffic_safety_seed0")
!python -m scripts.sync_drive_runs --drive-project "{DRIVE_PROJECT}" --to-drive --run-dir "{SAFETY_RUN}"

## 13. Pilot comparison and saved selection decision

In [ ]:
for pilot_run in [REFERENCE_RUN, SAFETY_RUN]:
    if run_status(pilot_run) != "complete":
        raise RuntimeError(f"Pilot is terminal or incomplete; report it without inventing another pilot: {pilot_run}")
    !python -m scripts.evaluate --run-dir "{pilot_run}" --model best --split validation --episodes 50 --densities 0.0 0.05 0.10 --prefix traffic_pilot validation.num_scenarios=50
    !python -m scripts.sync_drive_runs --drive-project "{DRIVE_PROJECT}" --to-drive --run-dir "{pilot_run}"
!python -m scripts.compare_runs --traffic-extension --select-pilots
SELECTION = json.loads((RUNS_DIR / "traffic_extension_selection.json").read_text())
print(json.dumps(SELECTION, indent=2))

## 14. Experiment 5: selected seed-1 confirmation

In [ ]:
SELECTED_VARIANT = SELECTION["selected_variant"]
pointer = f"sac_traffic_{SELECTED_VARIANT}_seed1"
action, CONFIRMATION_RUN = training_action(pointer)
if action == "start":
    !python -m scripts.train_curriculum --config configs/sac_traffic_curriculum.yaml --variant {SELECTED_VARIANT} --seed 1 --n-envs {N_ENVS} --vec-env {VEC_ENV} --progress
elif action == "resume":
    !python -m scripts.train_curriculum --config configs/sac_traffic_curriculum.yaml --variant {SELECTED_VARIANT} --seed 1 --n-envs {N_ENVS} --vec-env {VEC_ENV} --progress --run-dir "{CONFIRMATION_RUN}"
else:
    print(f"Confirmation action: {action}; run: {CONFIRMATION_RUN}")
CONFIRMATION_RUN = optional_latest(pointer)
!python -m scripts.sync_drive_runs --drive-project "{DRIVE_PROJECT}" --to-drive --run-dir "{CONFIRMATION_RUN}"

## 15. Final evaluation matrix

In [ ]:
if run_status(CONFIRMATION_RUN) != "complete":
    raise RuntimeError("The selected seed-1 confirmation must complete before final evaluation.")
SELECTED_SEED0_RUN = optional_latest(f"sac_traffic_{SELECTED_VARIANT}_seed0")
for adapted_run in [SELECTED_SEED0_RUN, CONFIRMATION_RUN]:
    !python -m scripts.evaluate --run-dir "{adapted_run}" --model best --split test --episodes 100 --densities 0.0 0.05 0.10 --prefix traffic_final
    !python -m scripts.sync_drive_runs --drive-project "{DRIVE_PROJECT}" --to-drive --run-dir "{adapted_run}"
!python -m scripts.evaluate_baseline --config configs/sac_traffic_curriculum.yaml --policy idm --split test --episodes 100 --densities 0.0 0.05 0.10 --prefix traffic_final_idm
!python -m scripts.evaluate_baseline --config configs/sac_traffic_curriculum.yaml --policy expert --split test --episodes 100 --densities 0.0 0.05 0.10 --prefix traffic_final_expert
for name in ["traffic_extension_idm", "traffic_extension_expert"]:
    baseline_run = optional_latest(name)
    !python -m scripts.sync_drive_runs --drive-project "{DRIVE_PROJECT}" --to-drive --run-dir "{baseline_run}"

## 16. Third-person videos

In [ ]:
from scripts.record_video import select_video_scenario
ADAPTED_CSV = SELECTED_SEED0_RUN / "eval" / "traffic_final_d005_episodes.csv"
MATCHED_SEED = select_video_scenario(ADAPTED_CSV, "first")
SOURCE_RUN = optional_latest("sac_phase2_curriculum_seed0")
!python -m scripts.record_video --run-dir "{SOURCE_RUN}" --model best --view chase --density 0.05 --seed {MATCHED_SEED} --output runs/videos/frozen_before_matched_chase.mp4
!python -m scripts.record_video --run-dir "{SELECTED_SEED0_RUN}" --model best --view chase --density 0.05 --seed {MATCHED_SEED} --output runs/videos/adapted_after_matched_chase.mp4
!python -m scripts.record_video --run-dir "{SELECTED_SEED0_RUN}" --model best --view chase --density 0.05 --episodes-csv "{ADAPTED_CSV}" --scenario-rule first_success --output runs/videos/adapted_representative_success_chase.mp4
if (~pd.read_csv(ADAPTED_CSV)["success"].astype(bool)).any():
    !python -m scripts.record_video --run-dir "{SELECTED_SEED0_RUN}" --model best --view chase --density 0.05 --episodes-csv "{ADAPTED_CSV}" --scenario-rule first_failure --output runs/videos/adapted_diagnostic_failure_chase.mp4
!python -m scripts.record_video --config configs/sac_traffic_curriculum.yaml --policy expert --view chase --density 0.05 --seed {MATCHED_SEED} --output runs/videos/expert_demonstration_chase.mp4

## 17. Final plots and tables

In [ ]:
!python -m scripts.compare_runs --traffic-extension
display(pd.read_csv(RUNS_DIR / "traffic_extension_comparison.csv"))

## 18. Build main and surrogate reports

In [ ]:
!latexmk -cd -pdf -interaction=nonstopmode -halt-on-error reports/main.tex
!latexmk -cd -pdf -interaction=nonstopmode -halt-on-error reports/surrogate_notes.tex

## 19. Final Drive synchronization

In [ ]:
for run_dir in [SELECTED_SEED0_RUN, CONFIRMATION_RUN]:
    !python -m scripts.sync_drive_runs --drive-project "{DRIVE_PROJECT}" --to-drive --run-dir "{run_dir}"
!python -m scripts.sync_drive_runs --drive-project "{DRIVE_PROJECT}" --project-artifacts-to-drive
print("Final artifacts persisted to", DRIVE_PROJECT)